# Endogenous Population Dynamics — Two-Timescale Replicator

**Goal**: Agents switch types based on relative P&L performance via two-timescale replicator dynamics.

$$\frac{dp_i}{dt} = \eta_{\text{fast}} \cdot p_i \cdot (\pi_i^{\text{short}} - \bar{\pi}^{\text{short}}) + \eta_{\text{slow}} \cdot p_i \cdot (\pi_i^{\text{long}} - \bar{\pi}^{\text{long}})$$

- **Fast timescale** ($\eta_{\text{fast}}$): retail-style herding on short-window P&L
- **Slow timescale** ($\eta_{\text{slow}}$): institutional rebalancing on long-window P&L

## Key Questions
1. Do emergent population cycles arise? (Value dominates → EMH → emotion enters → bubble → crash → cycle)
2. How does timescale separation (η_fast/η_slow ratio) affect dynamics?
3. Are there evolutionary stable strategies (fixed points on the simplex)?
4. What attractor types emerge as function of (η_fast, η_slow)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.tri as mtri

import sys
sys.path.insert(0, '..')

from src.agents import ModelParams
from src.simulation import simulate_market
from src.replicator import simulate_market_endogenous, ReplicatorParams
from src.visualization import ternary_coords

%matplotlib inline
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

params = ModelParams(T=10.0)
print(f'Simulation: {params.T} years, {params.n_steps} steps')

## 1. P&L Attribution by Agent Type

Verify: Type I profits in mean-reverting regimes, Type II profits in trending regimes.

In [ ]:
# Run with equal mix and moderate adaptation
rep = ReplicatorParams(eta_fast=0.5, eta_slow=0.05)
result = simulate_market_endogenous((0.33, 0.34, 0.33), params, rep, seed=42)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# P&L short-window
axes[0].plot(result['time'], result['pnl_short'][:, 0], 'b-', alpha=0.7, label='Type I (Value)')
axes[0].plot(result['time'], result['pnl_short'][:, 1], 'r-', alpha=0.7, label='Type II (Emotion)')
axes[0].plot(result['time'], result['pnl_short'][:, 2], 'gray', alpha=0.5, label='Type III (Noise)')
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].set_ylabel('Short-window P&L')
axes[0].set_title('P&L Attribution by Agent Type', fontsize=14, fontweight='bold')
axes[0].legend()

# P&L long-window
axes[1].plot(result['time'], result['pnl_long'][:, 0], 'b-', alpha=0.7, label='Type I')
axes[1].plot(result['time'], result['pnl_long'][:, 1], 'r-', alpha=0.7, label='Type II')
axes[1].plot(result['time'], result['pnl_long'][:, 2], 'gray', alpha=0.5, label='Type III')
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].set_ylabel('Long-window P&L')
axes[1].legend()

# Price
axes[2].plot(result['time'], result['price'], 'b-', linewidth=1, label='Price')
axes[2].plot(result['time'], result['fair_value'], 'k--', alpha=0.5, label='$V^*$')
axes[2].set_ylabel('Price')
axes[2].set_xlabel('Time (years)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 2. Population Trajectory on the Simplex

Single-path visualization of how population evolves over time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: time series of population fractions
axes[0].plot(result['time'], result['p1'], 'b-', linewidth=1.5, label='Type I (Value)')
axes[0].plot(result['time'], result['p2'], 'r-', linewidth=1.5, label='Type II (Emotion)')
axes[0].plot(result['time'], result['p3'], 'gray', linewidth=1.5, label='Type III (Noise)')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Population fraction')
axes[0].set_title('Population Dynamics Over Time', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
axes[0].legend()

# Right: trajectory on ternary simplex
ax = axes[1]
# Draw simplex
triangle = plt.Polygon(
    [ternary_coords(1,0,0), ternary_coords(0,1,0), ternary_coords(0,0,1)],
    fill=False, edgecolor='black', linewidth=2
)
ax.add_patch(triangle)

# Plot trajectory colored by time
x, y = ternary_coords(result['p1'], result['p2'], result['p3'])
scatter = ax.scatter(x, y, c=result['time'], cmap='viridis', s=0.5, alpha=0.6)
ax.plot(x[0], y[0], 'go', markersize=10, label='Start', zorder=5)
ax.plot(x[-1], y[-1], 'rs', markersize=10, label='End', zorder=5)

# Labels
ax.text(*ternary_coords(1,0,0), '\nType I\n(Value)', ha='center', va='top', fontsize=11, fontweight='bold')
ax.text(*ternary_coords(0,1,0), '\nType II\n(Emotion)', ha='center', va='top', fontsize=11, fontweight='bold')
tx, ty = ternary_coords(0,0,1)
ax.text(tx, ty+0.05, 'Type III\n(Noise)', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.colorbar(scatter, ax=ax, label='Time (years)')
ax.set_title('Population Trajectory on Simplex', fontsize=13, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 3. Phase Portrait: Population Flow Field

For fixed (η_fast, η_slow), show the average direction of population change at each point on the simplex.

In [ ]:
def compute_flow_field(eta_fast, eta_slow, n_grid=15, n_mc=10, T=5.0):
    """Compute average population velocity at each simplex point."""
    params_short = ModelParams(T=T)
    rep = ReplicatorParams(eta_fast=eta_fast, eta_slow=eta_slow)
    
    points = []
    velocities = []
    
    for i in range(n_grid + 1):
        for j in range(n_grid + 1 - i):
            k = n_grid - i - j
            p1 = max(i / n_grid, 0.02)
            p2 = max(j / n_grid, 0.02)
            p3 = max(k / n_grid, 0.02)
            total = p1 + p2 + p3
            p1, p2, p3 = p1/total, p2/total, p3/total
            
            dp_avg = np.zeros(3)
            for mc in range(n_mc):
                r = simulate_market_endogenous((p1, p2, p3), params_short, rep, seed=mc*100+i*20+j)
                # Average velocity = (final - initial) / T
                dp = r['population'][-1] - r['population'][0]
                dp_avg += dp / T
            dp_avg /= n_mc
            
            points.append((p1, p2, p3))
            velocities.append(dp_avg)
    
    return points, velocities


# Compute for moderate adaptation
print('Computing flow field (this may take a few minutes)...')
points, vels = compute_flow_field(eta_fast=1.0, eta_slow=0.1, n_grid=12, n_mc=5)
print('Done!')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Draw simplex
triangle = plt.Polygon(
    [ternary_coords(1,0,0), ternary_coords(0,1,0), ternary_coords(0,0,1)],
    fill=False, edgecolor='black', linewidth=2
)
ax.add_patch(triangle)

# Plot quiver field
for (p1, p2, p3), dp in zip(points, vels):
    x, y = ternary_coords(p1, p2, p3)
    # Convert dp to ternary coordinates
    dx, dy = ternary_coords(p1 + dp[0], p2 + dp[1], p3 + dp[2])
    dx -= x
    dy -= y
    speed = np.sqrt(dx**2 + dy**2)
    if speed > 1e-6:
        ax.arrow(x, y, dx*2, dy*2, head_width=0.01, head_length=0.005,
                 fc='darkblue', ec='darkblue', alpha=0.6)

# Labels
ax.text(*ternary_coords(1,0,0), '\nType I\n(Value)', ha='center', va='top', fontsize=12, fontweight='bold')
ax.text(*ternary_coords(0,1,0), '\nType II\n(Emotion)', ha='center', va='top', fontsize=12, fontweight='bold')
tx, ty = ternary_coords(0,0,1)
ax.text(tx, ty+0.05, 'Type III\n(Noise)', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_title('Population Flow Field\n$\\eta_{fast}=1.0$, $\\eta_{slow}=0.1$',
             fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

plt.tight_layout()
plt.show()

## 4. Static vs Single-Timescale vs Two-Timescale Comparison

Compare the three model variants to see what two-timescale dynamics add.

In [ ]:
p0 = (0.33, 0.34, 0.33)
seed = 7

# Static (no adaptation)
static_result = simulate_market(p0, params, seed=seed, force_python=True)

# Single-timescale (fast only)
single_result = simulate_market_endogenous(
    p0, params, ReplicatorParams(eta_fast=1.0, eta_slow=0.0), seed=seed
)

# Two-timescale
dual_result = simulate_market_endogenous(
    p0, params, ReplicatorParams(eta_fast=1.0, eta_slow=0.1), seed=seed
)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Price comparison
ax = axes[0, 0]
ax.plot(static_result['time'], static_result['price'], 'b-', alpha=0.7, label='Static')
ax.plot(single_result['time'], single_result['price'], 'r-', alpha=0.7, label='Single-timescale')
ax.plot(dual_result['time'], dual_result['price'], 'g-', alpha=0.7, label='Two-timescale')
ax.plot(static_result['time'], static_result['fair_value'], 'k--', alpha=0.3, label='$V^*$')
ax.set_title('Price Paths', fontweight='bold')
ax.set_ylabel('Price')
ax.legend(fontsize=9)

# Population - single timescale
ax = axes[0, 1]
ax.plot(single_result['time'], single_result['p1'], 'b-', label='Value')
ax.plot(single_result['time'], single_result['p2'], 'r-', label='Emotion')
ax.plot(single_result['time'], single_result['p3'], 'gray', label='Noise')
ax.set_title('Single-Timescale Population', fontweight='bold')
ax.set_ylabel('Fraction')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# Population - two timescale
ax = axes[1, 0]
ax.plot(dual_result['time'], dual_result['p1'], 'b-', label='Value')
ax.plot(dual_result['time'], dual_result['p2'], 'r-', label='Emotion')
ax.plot(dual_result['time'], dual_result['p3'], 'gray', label='Noise')
ax.set_title('Two-Timescale Population', fontweight='bold')
ax.set_ylabel('Fraction')
ax.set_xlabel('Time (years)')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# Mispricing comparison
ax = axes[1, 1]
ax.plot(static_result['time'], static_result['mispricing'], 'b-', alpha=0.7, label='Static')
ax.plot(single_result['time'], single_result['mispricing'], 'r-', alpha=0.7, label='Single')
ax.plot(dual_result['time'], dual_result['mispricing'], 'g-', alpha=0.7, label='Two-timescale')
ax.axhline(0, color='k', linewidth=0.5)
ax.set_title('Mispricing $\\log(P/V^*)$', fontweight='bold')
ax.set_xlabel('Time (years)')
ax.legend(fontsize=9)

plt.suptitle('Static vs Single-Timescale vs Two-Timescale Dynamics\n$\\mathbf{p}_0 = (0.33, 0.34, 0.33)$',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Two-Timescale Bifurcation Diagram

Sweep (η_fast, η_slow) and classify the attractor type at each point.

In [ ]:
def classify_attractor(result, tail_frac=0.3):
    """Classify the attractor type from a simulation result.
    
    Returns:
        dict with metrics for classification.
    """
    pop = result['population']
    n = len(pop)
    tail = int(n * (1 - tail_frac))
    pop_tail = pop[tail:]
    
    # Population variance (low = fixed point, high = oscillations/chaos)
    pop_var = np.mean(np.var(pop_tail, axis=0))
    
    # Final population (where does it converge?)
    p_final = pop[-1]
    
    # Mispricing statistics
    misp_tail = result['mispricing'][tail:]
    mean_misp = np.mean(np.abs(misp_tail))
    
    # Excess volatility
    rets = result['returns'][tail:]
    price_vol = np.std(rets) * np.sqrt(252)
    fv_rets = np.diff(np.log(result['fair_value'][tail:]))
    fv_vol = np.std(fv_rets) * np.sqrt(252)
    excess_vol = price_vol / fv_vol if fv_vol > 1e-10 else np.nan
    
    return {
        'pop_var': pop_var,
        'p_final': p_final,
        'mean_mispricing': mean_misp,
        'excess_vol': excess_vol,
    }

In [ ]:
# Sweep eta_fast x eta_slow
eta_fast_range = np.array([0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0])
eta_slow_range = np.array([0.0, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5])

p0 = (0.33, 0.34, 0.33)
n_mc = 10

bifurcation_data = []

print(f'Sweeping {len(eta_fast_range)} x {len(eta_slow_range)} = {len(eta_fast_range)*len(eta_slow_range)} parameter pairs...')

for ef in eta_fast_range:
    for es in eta_slow_range:
        metrics_list = []
        for mc in range(n_mc):
            rep = ReplicatorParams(eta_fast=ef, eta_slow=es)
            result = simulate_market_endogenous(p0, params, rep, seed=mc)
            metrics = classify_attractor(result)
            metrics_list.append(metrics)
        
        # Average metrics
        avg_metrics = {
            'eta_fast': ef,
            'eta_slow': es,
            'pop_var': np.mean([m['pop_var'] for m in metrics_list]),
            'mean_mispricing': np.mean([m['mean_mispricing'] for m in metrics_list]),
            'excess_vol': np.nanmean([m['excess_vol'] for m in metrics_list]),
            'p1_final': np.mean([m['p_final'][0] for m in metrics_list]),
            'p2_final': np.mean([m['p_final'][1] for m in metrics_list]),
        }
        bifurcation_data.append(avg_metrics)

print('Done!')

In [ ]:
# Plot bifurcation diagram: 2x2 heatmaps
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

n_fast = len(eta_fast_range)
n_slow = len(eta_slow_range)

for ax, metric, title, cmap in zip(
    axes.flat,
    ['pop_var', 'mean_mispricing', 'excess_vol', 'p1_final'],
    ['Population Variance', 'Mean |Mispricing|', 'Excess Volatility', 'Final Value Trader Fraction'],
    ['hot', 'RdYlGn_r', 'hot', 'RdYlGn'],
):
    Z = np.array([d[metric] for d in bifurcation_data]).reshape(n_fast, n_slow)
    
    im = ax.imshow(Z, origin='lower', aspect='auto', cmap=cmap,
                   extent=[eta_slow_range[0], eta_slow_range[-1],
                           eta_fast_range[0], eta_fast_range[-1]])
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('$\\eta_{slow}$')
    ax.set_ylabel('$\\eta_{fast}$')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Bifurcation Diagram: $(\\eta_{fast}, \\eta_{slow})$ Sweep\n'
             '$\\mathbf{p}_0 = (0.33, 0.34, 0.33)$, $T = 10$ years',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Basin of Attraction Analysis

For fixed (η_fast, η_slow), sweep initial conditions across the simplex and observe where each trajectory converges.

In [ ]:
# Fixed adaptation rates
rep_basin = ReplicatorParams(eta_fast=1.0, eta_slow=0.1)
n_grid_basin = 15
n_mc_basin = 5

basin_results = []

print(f'Sweeping initial conditions on simplex ({n_grid_basin+1} x grid)...')

for i in range(n_grid_basin + 1):
    for j in range(n_grid_basin + 1 - i):
        k = n_grid_basin - i - j
        p1 = max(i / n_grid_basin, 0.02)
        p2 = max(j / n_grid_basin, 0.02)
        p3 = max(k / n_grid_basin, 0.02)
        total = p1 + p2 + p3
        p1, p2, p3 = p1/total, p2/total, p3/total
        
        finals = []
        for mc in range(n_mc_basin):
            r = simulate_market_endogenous((p1, p2, p3), params, rep_basin, seed=mc*100+i*20+j)
            finals.append(r['population'][-1])
        
        avg_final = np.mean(finals, axis=0)
        basin_results.append({
            'p1_init': p1, 'p2_init': p2, 'p3_init': p3,
            'p1_final': avg_final[0], 'p2_final': avg_final[1], 'p3_final': avg_final[2],
        })

print(f'Done! {len(basin_results)} initial conditions tested.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, key, title in zip(
    axes,
    ['p1_final', 'p2_final', 'p3_final'],
    ['Final Value Fraction', 'Final Emotion Fraction', 'Final Noise Fraction'],
):
    p1_arr = np.array([r['p1_init'] for r in basin_results])
    p2_arr = np.array([r['p2_init'] for r in basin_results])
    p3_arr = np.array([r['p3_init'] for r in basin_results])
    vals = np.array([r[key] for r in basin_results])
    
    x, y = ternary_coords(p1_arr, p2_arr, p3_arr)
    triang = mtri.Triangulation(x, y)
    
    tcf = ax.tricontourf(triang, vals, levels=20, cmap='RdYlGn', vmin=0, vmax=1)
    
    triangle = plt.Polygon(
        [ternary_coords(1,0,0), ternary_coords(0,1,0), ternary_coords(0,0,1)],
        fill=False, edgecolor='black', linewidth=2
    )
    ax.add_patch(triangle)
    
    ax.text(*ternary_coords(1,0,0), '\nI', ha='center', va='top', fontsize=11, fontweight='bold')
    ax.text(*ternary_coords(0,1,0), '\nII', ha='center', va='top', fontsize=11, fontweight='bold')
    tx, ty = ternary_coords(0,0,1)
    ax.text(tx, ty+0.03, 'III', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.colorbar(tcf, ax=ax, shrink=0.8)
    ax.set_title(title, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

plt.suptitle('Basins of Attraction\n$\\eta_{fast}=1.0$, $\\eta_{slow}=0.1$, $T=10$ years',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Timescale Separation Analysis

What happens as η_fast/η_slow → ∞? Fix η_slow, increase η_fast.

In [ ]:
eta_slow_fixed = 0.05
ratios = [1, 2, 5, 10, 20, 50, 100]
p0 = (0.33, 0.34, 0.33)

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for idx, ratio in enumerate(ratios):
    ef = eta_slow_fixed * ratio
    rep = ReplicatorParams(eta_fast=ef, eta_slow=eta_slow_fixed)
    result = simulate_market_endogenous(p0, params, rep, seed=42)
    
    ax = axes.flat[idx]
    ax.plot(result['time'], result['p1'], 'b-', linewidth=1, label='Value')
    ax.plot(result['time'], result['p2'], 'r-', linewidth=1, label='Emotion')
    ax.plot(result['time'], result['p3'], 'gray', linewidth=1, label='Noise')
    ax.set_title(f'$\\eta_f/\\eta_s = {ratio}$\n($\\eta_f={ef:.2f}$)', fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1)
    if idx == 0:
        ax.legend(fontsize=8)
    if idx >= 4:
        ax.set_xlabel('Time (years)')

# Hide last subplot
axes.flat[-1].set_visible(False)

plt.suptitle(f'Timescale Separation: $\\eta_{{slow}} = {eta_slow_fixed}$, varying $\\eta_{{fast}}/\\eta_{{slow}}$',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()